# 情報基礎 対面3 ── 学習データの外側としきい値

受講者が提出した手書きの数字で，数字を識別するプログラム（入力 64個 → 中間層 16個 → 出力 10個）を学習させる．学習に使っていない数字と，数字でないもの（アルファベットと模様）を入力し，出力される確率を調べる．

**進め方**

- 上のセルから順に，セルの左の ▶（実行ボタン）を押す．コードは書き換えない（3 の `show(" ")` の " " の中と，5 の T の値だけ書き換える）
- `NameError` が出たら，1 のセルから順に実行し直す
- ほかのエラーが出たら，まず最後の1行を読む．エラーの種類と内容が表示される

## 1 データの読み込み

次のセルを実行する．受講者が提出した 8×8 の絵を読み込む．

- 学習データ：学習に使う数字
- テストデータ：学習に使わず，正解率を調べるために残した数字（0〜9 の各3個）
- 数字でないもの：アルファベットと模様

In [ ]:
import urllib.request
import numpy as np
import matplotlib.pyplot as plt

URL = "https://raw.githubusercontent.com/kazumichiINOUE/info-basics-notebooks/7ee53a7c8b8f1dfcaf74a7515bc6f65927dc963e/data/k1_g1.txt"


def to_bits(h):
    """16進数16文字を，64個の 0 と 1 の列にする"""
    v = int(h, 16)
    return np.array([v >> (63 - i) & 1 for i in range(64)], dtype=float)


data = {"train": [], "test": [], "other": []}
for line in urllib.request.urlopen(URL).read().decode("utf-8").splitlines():
    if line and not line.startswith("#"):
        kind, label, h = line.split()
        data[kind].append((label, h))

train_x = np.array([to_bits(h) for _, h in data["train"]])
train_t = np.array([int(a) for a, _ in data["train"]])
test_x = np.array([to_bits(h) for _, h in data["test"]])
test_t = np.array([int(a) for a, _ in data["test"]])
other_x = np.array([to_bits(h) for _, h in data["other"]])
other_label = [a for a, _ in data["other"]]
print("学習データ　　　", len(train_x), "個")
print("テストデータ　　", len(test_x), "個")
print("数字でないもの　", len(other_x), "個")

## 2 学習

次のセルを実行する．入力 64個 → 中間層 16個 → 出力 10個 のプログラムを，学習データで学習させる．数秒かかる．

- 出力の10個は，0〜9 のそれぞれである確率．10個の和は 1
- パラメータの初期値は乱数．乱数の種を固定してあるので，全員が同じ結果になる
- 学習データ全体で 3000エポック学習させる

In [ ]:
rng = np.random.default_rng(0)       # 乱数の種
W1 = rng.normal(0, 0.3, (64, 16))    # 入力 → 中間層の重み
b1 = np.zeros(16)                    # 中間層のバイアス
W2 = rng.normal(0, 0.3, (16, 10))    # 中間層 → 出力の重み
b2 = np.zeros(10)                    # 出力のバイアス
lr = 0.5                             # 学習率


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def predict(x):
    """入力（64個の 0 と 1）から，中間層の値と 10個の確率を計算する"""
    h = sigmoid(x @ W1 + b1)
    z = h @ W2 + b2
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return h, e / e.sum(axis=-1, keepdims=True)


target = np.eye(10)[train_t]         # 正解ラベルを 1，ほかを 0 にした表
for epoch in range(3000):
    h, y = predict(train_x)
    d = (y - target) / len(train_x)  # 出力と正解の差
    dh = d @ W2.T * h * (1 - h)
    W2 -= lr * h.T @ d
    b2 -= lr * d.sum(axis=0)
    W1 -= lr * train_x.T @ dh
    b1 -= lr * dh.sum(axis=0)

_, y_train = predict(train_x)
_, y_test = predict(test_x)
print("学習データでの正解率　", f"{(y_train.argmax(axis=1) == train_t).mean():.0%}")
print("テストデータでの正解率", f"{(y_test.argmax(axis=1) == test_t).mean():.0%}")

## 3 1個の絵の確率

次のセルを実行する．絵と，その 10個の確率を表示する関数を定義するだけなので，何も表示されない．

In [ ]:
def show(h):
    """16進数の絵と，その 10個の確率を表示する"""
    x = to_bits(h)
    _, y = predict(x)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3), gridspec_kw={"width_ratios": [1, 2]})
    ax1.imshow(x.reshape(8, 8), cmap="Greys", vmin=0, vmax=1)
    ax1.set_xticks([])
    ax1.set_yticks([])
    ax2.bar(range(10), y)
    ax2.set_xticks(range(10))
    ax2.set_ylim(0, 1)
    plt.show()
    print("確率が最大のラベル", y.argmax(), "　確率", round(float(y.max()), 3))

[「8×8グリッド」のページ](https://webclass.gks.saitama-u.ac.jp/webclass/txtbk_frame.php?set_contents_id=24b0ac8469c2fdcdd3fd3b0ab2dd0daf&language=JAPANESE)で絵を描き，16進数をコピーする．次のセルの " " の中に貼って実行する．左に絵，右に 10個の確率が表示される．例の `3C66666666663C00` は「0」の絵．

In [ ]:
show("3C66666666663C00")

## 4 数字でないもの

次のセルを実行する．前回提出されたアルファベットと模様から 24個を表示する．絵の上は「元のラベル → 確率が最大のラベル（その確率）」．

In [ ]:
_, y_other = predict(other_x)
pick = np.linspace(0, len(other_x) - 1, 24).astype(int)
fig, axes = plt.subplots(3, 8, figsize=(12, 5))
for ax, i in zip(axes.flat, pick):
    ax.imshow(other_x[i].reshape(8, 8), cmap="Greys", vmin=0, vmax=1)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f"{other_label[i]} → {y_other[i].argmax()} ({y_other[i].max():.2f})", fontsize=10)
plt.show()
print("数字でないもの", len(other_x), "個のうち，最大の確率が 0.9 以上のもの", (y_other.max(axis=1) >= 0.9).sum(), "個")

## 5 しきい値

規則：最大の確率が T 以上なら，そのラベルを出力する．T 未満なら「わからない」を出力する（棄却する）．

次のセルの T に，自分で決めた値（0〜1）を入れて実行する．次の2つの割合が表示される．

- テストデータの数字のうち，正しいラベルを出力した割合
- 数字でないもののうち，棄却した割合

In [ ]:
T = 0.9   # しきい値

answered = (y_test.max(axis=1) >= T) & (y_test.argmax(axis=1) == test_t)
rejected = y_other.max(axis=1) < T
print("T =", T)
print("テストデータの数字のうち，正しいラベルを出力した割合", f"{answered.mean():.0%}", f"（{answered.sum()}/{len(test_x)}）")
print("数字でないもののうち，棄却した割合　　　　　　　　　", f"{rejected.mean():.0%}", f"（{rejected.sum()}/{len(other_x)}）")

## 6 T を変えたとき

次のセルを実行する．T を変えたときの2つの割合を，表にして表示する．

In [ ]:
print("  T　　正しいラベルを出力（数字）　棄却（数字でないもの）")
for T in [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99]:
    answered = (y_test.max(axis=1) >= T) & (y_test.argmax(axis=1) == test_t)
    rejected = y_other.max(axis=1) < T
    print(f"{T:5.2f}　　{answered.mean():4.0%}　　　　　　　　　　 {rejected.mean():4.0%}")